# CHATBOT CIMA | Actividad 8 — Registro de observabilidad en MLflow

**Objetivo:** importar a MLflow las evidencias **SIMULADAS OFFLINE** de GitHub Actions y obtener capturas de experimentos, métricas, parámetros y artefactos. **No monitorea Render en tiempo real ni modifica el modelo.**

**Requisitos:** runtime CPU de Google Colab (no se requiere GPU), conexión para instalar dependencias y clonar GitHub, y el artefacto `cima-a8-simulacion.zip` descargado desde la **última ejecución verde** de GitHub Actions. El entorno de Colab es temporal; exporta el almacén al finalizar.

**Evidencias que debes capturar:** pantalla de runs MLflow, métricas del incidente de latencia, métricas de calidad ficticia y lista de artefactos. No subir información personal o bancaria.

## Celda 1 — Instalación de MLflow

Se instala MLflow sobre Python del entorno en lugar de fijar versiones de scikit-learn del proyecto. **No instalar ni ejecutar los modelos de Fase II aquí.**

In [1]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow>=3.5,<4'], check=True)
import mlflow
print('Python:',sys.version.split()[0], '| MLflow:',mlflow.__version__)

Python: 3.13.18 | MLflow: 3.18.0


## Celda 2 — Obtener el código publicado en GitHub

Verifica antes que `Actividad8/src/mlflow_registro.py` ya esté publicado. Si no existe, súbelo desde el paquete de actualización.

In [2]:
from pathlib import Path
repo=Path('/content/deep-learning-experiments')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/alfredosalazar-ai/deep-learning-experiments.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
project=repo/'Actividad8'
script=project/'src'/'mlflow_registro.py'
assert script.is_file(),f'Archivo faltante en GitHub: {script}. Publícalo antes de continuar.'
print('Código disponible:',script)

Código disponible: /content/deep-learning-experiments/Actividad8/src/mlflow_registro.py


## Celda 3 — Cargar el ZIP de la ejecución exitosa

En GitHub → Actions → última ejecución **Success** → Artifacts → descarga `cima-a8-simulacion`. **Sube el archivo ZIP, sin descomprimirlo.** Es la fuente de verdad de esta demostración.

In [3]:
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1,'Sube únicamente el ZIP de evidencia.'
name=next(iter(uploaded))
assert name.lower().endswith('.zip'),'Se requiere un archivo ZIP.'
zip_path=Path('/content')/name
if not zip_path.exists():
    zip_path.write_bytes(uploaded[name])
print('Artefacto recibido:',zip_path.name, '| tamaño:',zip_path.stat().st_size,'bytes')

Saving cima-a8-simulacion.zip to cima-a8-simulacion.zip
Artefacto recibido: cima-a8-simulacion.zip | tamaño: 101150 bytes


## Celda 4 — Registrar cinco runs en MLflow

Se registran 3 incidentes (uno por run), 1 simulación de calidad ficticia y 1 run de alertas/runbooks. `--workflow-run` y `--commit` son **referencias declaradas por el usuario**, no verificación automática del enlace de GitHub.

In [6]:
import os
tracking_dir=Path('/content/cima_mlflow')
cmd=[sys.executable,'-m','src.mlflow_registro','--input',str(zip_path),'--tracking-dir',str(tracking_dir),'--workflow-run','4','--commit','c0d6991']

# Clonamos las variables de entorno actuales e inyectamos la requerida por MLflow
env_vars = os.environ.copy()
env_vars['MLFLOW_ALLOW_FILE_STORE'] = 'true'

try:
    # Ejecutamos el script pasando la variable de entorno
    result = subprocess.run(cmd,cwd=str(project),check=True,capture_output=True,text=True,env=env_vars)
    print('Registro concluido. Los resultados son SIMULACIONES OFFLINE.')
except subprocess.CalledProcessError as e:
    print("--- ERROR DURANTE LA EJECUCIÓN DEL SCRIPT ---")
    print("Código de salida:", e.returncode)
    print("\n--- SALIDA ESTÁNDAR (STDOUT) ---")
    print(e.stdout)
    print("\n--- DETALLE DEL ERROR (STDERR) ---")
    print(e.stderr)
    raise e

Registro concluido. Los resultados son SIMULACIONES OFFLINE.


## Celda 5 — Verificación de runs y métricas (captura 1)

La tabla se obtiene **del almacén MLflow registrado en esta sesión**, no de valores insertados manualmente.

In [8]:
import os
# Permitir el uso del backend local de archivos en MLflow interactivo
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'

from IPython.display import display
mlflow.set_tracking_uri((tracking_dir/'mlruns').as_uri())
runs=mlflow.search_runs(experiment_names=['CIMA_Actividad8_Observabilidad_Simulada'])
assert len(runs)>=5,f'Se esperaban al menos 5 runs; hay {len(runs)}'
campos=[c for c in ['tags.mlflow.runName','tags.tipo_run','metrics.latencia_p95_ms','metrics.drift_psi','metrics.error_5xx_rate','metrics.accuracy','metrics.f1_macro','metrics.runbooks_recuperacion_verificada'] if c in runs.columns]
display(runs[campos].fillna('—'))
print('Runs de esta sesión:',len(runs), '| Fuente: SIMULACION_OFFLINE')

,tags.mlflow.runName,tags.tipo_run,metrics.latencia_p95_ms,metrics.drift_psi,metrics.error_5xx_rate,metrics.accuracy,metrics.f1_macro,metrics.runbooks_recuperacion_verificada
0,ALERTAS_SLO_y_runbooks,ALERTAS,—,—,—,—,—,3.0
1,CALIDAD_predicciones_ficticias,CALIDAD_FICTICIA,—,—,—,0.6875,0.6968,—
2,INC-03_incidente_offline,INCIDENTE,—,—,0.0,—,—,—
3,INC-02_incidente_offline,INCIDENTE,—,0.0,0.0,—,—,—
4,INC-01_incidente_offline,INCIDENTE,520.0,—,0.0,—,—,—


Runs de esta sesión: 5 | Fuente: SIMULACION_OFFLINE


## Celda 6 — Mostrar la interfaz MLflow (capturas 2 a 4)

Abre la ventana de MLflow y captura la pestaña **Runs**, las métricas de `INC-01_incidente_offline`, el run de calidad y los artefactos adjuntos.

En Colab puede abrirse una ventana mediante el proxy del kernel. Si la interfaz no carga, utiliza la tabla de la celda anterior como evidencia y conserva la salida del script. No expongas MLflow en un servidor público.

In [13]:

import mlflow
import pandas as pd
from IPython.display import display

# Utilizar el almacenamiento MLflow de esta sesión.
mlflow.set_tracking_uri(
    (tracking_dir / "mlruns").as_uri()
)

cliente = mlflow.tracking.MlflowClient()

experimento = cliente.get_experiment_by_name(
    "CIMA_Actividad8_Observabilidad_Simulada"
)

assert experimento is not None, "Experimento no encontrado"

runs = cliente.search_runs(
    experiment_ids=[experimento.experiment_id],
    max_results=100
)

assert len(runs) >= 5, "Se esperaban al menos 5 runs"

print("=" * 65)
print("CHATBOT CIMA - ACTIVIDAD 8")
print("EVIDENCIA DE EXPERIMENTOS MLFLOW")
print("=" * 65)
print("Experimento:", experimento.name)
print("Runs encontrados:", len(runs))
print("Origen: SIMULACIONES OFFLINE")
print()

resumen = []
for run in runs:
    resumen.append({
        "Run": run.data.tags.get("mlflow.runName", ""),
        "Tipo": run.data.tags.get("tipo_run", ""),
        "Estado": run.info.status,
        "Metricas": len(run.data.metrics),
        "Parametros": len(run.data.params),
        "Artefactos": len(
            cliente.list_artifacts(run.info.run_id)
        )
    })

display(pd.DataFrame(resumen))

for run in runs:
    print("\n" + "=" * 65)
    print("RUN:", run.data.tags.get("mlflow.runName"))
    print("ID:", run.info.run_id)
    print("ESTADO:", run.info.status)

    print("\nMETRICAS")
    display(pd.DataFrame(
        list(run.data.metrics.items()),
        columns=["Metrica", "Valor"]
    ))

    print("\nPARAMETROS")
    display(pd.DataFrame(
        list(run.data.params.items()),
        columns=["Parametro", "Valor"]
    ))

    print("\nARTEFACTOS")
    for archivo in cliente.list_artifacts(run.info.run_id):
        print(" -", archivo.path)


CHATBOT CIMA - ACTIVIDAD 8
EVIDENCIA DE EXPERIMENTOS MLFLOW
Experimento: CIMA_Actividad8_Observabilidad_Simulada
Runs encontrados: 5
Origen: SIMULACIONES OFFLINE



,Run,Tipo,Estado,Metricas,Parametros,Artefactos
0,ALERTAS_SLO_y_runbooks,ALERTAS,FINISHED,10,0,2
1,CALIDAD_predicciones_ficticias,CALIDAD_FICTICIA,FINISHED,7,0,2
2,INC-03_incidente_offline,INCIDENTE,FINISHED,4,2,2
3,INC-02_incidente_offline,INCIDENTE,FINISHED,5,2,2
4,INC-01_incidente_offline,INCIDENTE,FINISHED,5,2,2



RUN: ALERTAS_SLO_y_runbooks
ID: 3ab5126e8fec458694d7ee5499952e7a
ESTADO: FINISHED

METRICAS


,Metrica,Valor
0,primera_emision,1.0
1,runbooks_registrados,3.0
2,runbooks_recuperacion_verificada,3.0
3,segunda_emision_2min,0.0
4,alertas_generadas,5.0
5,emision_31min,1.0
6,alertas_suprimidas_estado_reutilizado,5.0
7,suprimidas_2min,1.0
8,cooldown_minutos,30.0
9,alertas_emitidas_estado_reutilizado,0.0



PARAMETROS


,Parametro,Valor


... (13 outputs left out)

,Metrica,Valor
0,error_5xx_rate,0.0
1,recuperacion_simulada,1.0
2,latencia_p95_ms,520.0
3,disponibilidad,1.0
4,alertas_activas,0.0



PARAMETROS


,Parametro,Valor
0,escenario,Latencia artificial
1,n_ventana,50



ARTEFACTOS
 - fuente
 - graficos


## Celda 7 — Respaldo antes de cerrar Colab

MLflow almacena los runs en el entorno temporal; descarga el respaldo para no perder el registro. **No subir el almacén completo a un repositorio público sin revisar su contenido.** En GitHub basta publicar capturas revisadas y el resumen JSON.

In [14]:

from pathlib import Path
import json
import pandas as pd
from google.colab import files
from mlflow.tracking import MlflowClient

# Carpeta de evidencias académicas.
salida = Path("/content/cima_a8_mlflow_evidencias")
salida.mkdir(parents=True, exist_ok=True)

cliente = MlflowClient()
experimento = cliente.get_experiment_by_name(
    "CIMA_Actividad8_Observabilidad_Simulada"
)
assert experimento is not None

runs = cliente.search_runs(
    experiment_ids=[experimento.experiment_id],
    max_results=100
)
assert len(runs) >= 5

registros = []
detalle = []

for run in runs:
    nombre = run.data.tags.get("mlflow.runName", "")
    tipo = run.data.tags.get("tipo_run", "")

    registro = {
        "run_id": run.info.run_id,
        "nombre": nombre,
        "tipo": tipo,
        "estado": run.info.status,
        **{
            "metrica_" + k: v
            for k, v in run.data.metrics.items()
        },
        **{
            "parametro_" + k: v
            for k, v in run.data.params.items()
        }
    }
    registros.append(registro)

    detalle.append({
        "run_id": run.info.run_id,
        "nombre": nombre,
        "tipo": tipo,
        "estado": run.info.status,
        "metricas": run.data.metrics,
        "parametros": run.data.params,
        "tags": run.data.tags,
        "artefactos": [
            a.path for a in
            cliente.list_artifacts(run.info.run_id)
        ]
    })

pd.DataFrame(registros).to_csv(
    salida / "mlflow_resumen_runs.csv",
    index=False,
    encoding="utf-8-sig"
)

(salida / "mlflow_detalle_runs.json").write_text(
    json.dumps(
        {
            "origen": "SIMULACIONES_OFFLINE",
            "experimento": experimento.name,
            "total_runs": len(runs),
            "runs": detalle
        },
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

# Comprimir únicamente los resultados exportados.
import shutil

archivo_zip = shutil.make_archive(
    "/content/cima_a8_mlflow_evidencias",
    "zip",
    root_dir=salida
)

print("Experimentos exportados:", len(runs))
print("Archivo generado:", archivo_zip)

files.download(archivo_zip)


Experimentos exportados: 5
Archivo generado: /content/cima_a8_mlflow_evidencias.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretación académica

Los resultados **son datos artificiales** reutilizados de GitHub Actions. MLflow demuestra que tenemos trazabilidad de los experimentos: parámetros, métricas, runs y artefactos asociados. No demuestra que exista observabilidad continua ni accuracy real de CIMA.

**Checklist de evidencia:** 5 runs visibles; INC-01 presenta latencia antes/durante/después; calidad ficticia indica caída de accuracy/F1; alertas y runbooks tienen artefactos; resumen JSON incluye hash SHA-256 y ubicación de origen. Para trazas distribuidas reales haría falta instrumentar el backend FastAPI, trabajo fuera de este laboratorio.

Fuente: [MLflow Tracking](https://mlflow.org/docs/latest/tracking/) · [Proyecto GitHub](https://github.com/alfredosalazar-ai/deep-learning-experiments/tree/main/Actividad8).